## Data Processing

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
# file directories

file_path = Path("../data/Global Factor Data.parquet")
FEATURES_RAW = Path("../data/features_clean.parquet")
TARGET_FILE = Path("../data/target.parquet")
FEATURES_FLAG = Path("../data/features_with_flags.parquet")
FEATURES_FINAL = Path("../data/features_ranked.parquet")

# Read full parquet file into Arrow Table
table = pq.read_table(file_path)
df = table.to_pandas()

display(df.head())

schema = pq.read_schema(file_path)
metadata = pq.read_metadata(file_path)
n_rows = metadata.num_rows
n_columns = len(schema.names)

print(f"Row count:{n_rows}, Column count: {n_columns}")

In [ ]:
parquet_file = pq.ParquetFile(file_path)
results = []

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):

    batch_table = pa.Table.from_batches([batch])
    df_batch = batch_table.to_pandas()

    nan_counts = df_batch.isnull().sum(axis=1)
    results.append(nan_counts)

    print(f"Batch {i + 1} processed | rows: {len(df_batch)}")

nan_per_row = pd.concat(results, ignore_index=True)
print(f"Total rows: {len(nan_per_row)}")
print(f"Rows with zero NaNs: {(nan_per_row == 0).sum()}")
print(f"Rows with at least one NaN: {(nan_per_row > 0).sum()}")
print(f"Average NaNs per row: {nan_per_row.mean():.2f}")
print(f"Max NaNs in a single row: {nan_per_row.max()}")

In [ ]:
raw_columns = schema.names
raw_n_rows  = metadata.num_rows

COLUMNS_TO_REMOVE: set[str] = {

    # Identifiers and keys
    "permno", "permco", "eom",

    # data-quality flags and others
    "obs_main", "exch_main", "primary_sec", "common", "size_grp",
    "ret_lag_dif", "adjfct", "bidask", "comp_tpci", "crsp_shrcd",
    "comp_exchg", "crsp_exchcd", "source_crsp", "curcd",

    # Target variable (isolated separately)
    "ret_exc_lead1m",

    # returns (direct look-ahead risk)
    "ret_exc", "ret", "ret_local",

    # Raw unscaled accounting numbers (dimensionless variants are retained)
    "enterprise_value", "book_equity", "assets", "sales", "net_income",

    # Exchange rates and industry codes
    "fx", "gics", "naics", "sic", "ff49",

    # Raw price and volume data
    "prc", "prc_local", "prc_high", "prc_low", "shares", "tvol",
}

TARGET_COLUMN = "ret_exc_lead1m"

# Partition the raw columns into feature and target lists. The feature list
# preserves the original parquet ordering for column-projection efficiency
feature_cols = [c for c in raw_columns if c not in COLUMNS_TO_REMOVE and c != TARGET_COLUMN]
target_cols = [TARGET_COLUMN] if TARGET_COLUMN in raw_columns else []

print(f"Columns removed: {len(COLUMNS_TO_REMOVE & set(raw_columns))}")
print(f"Features kept: {len(feature_cols)}")
print(f"Target present: {bool(target_cols)}")

In [ ]:
# Stream the raw parquet in batches, projecting onto feature and target columns
# and writing to separate output files. Lazy writer creation ensures the output
# schemas are inferred from the actual projected batch.

parquet_file = pq.ParquetFile(file_path)
feature_writer = None
target_writer = None
total_rows = 0

try:
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):
        batch_table = pa.Table.from_batches([batch])

        feature_table = batch_table.select(feature_cols)
        if feature_writer is None:
            feature_writer = pq.ParquetWriter(FEATURES_RAW, feature_table.schema, compression="snappy")
        feature_writer.write_table(feature_table)

        if target_cols:
            target_table = batch_table.select(target_cols)
            if target_writer is None:
                target_writer = pq.ParquetWriter(TARGET_FILE, target_table.schema, compression="snappy")
            target_writer.write_table(target_table)

        total_rows += batch_table.num_rows
        print(f"Batch {i + 1:2d}  rows ={batch_table.num_rows:>8,}  cumulative ={total_rows:>10,}")
finally:
    if feature_writer is not None:
        feature_writer.close()
    if target_writer is not None:
        target_writer.close()

### K0 and K1 groups

In [ ]:
# K1 patterns identify slow-moving accounting characteristics that receive
# the annual lag structure at months {12, 24, 36, 48, 60} in section 1.5.
k1_factors: tuple[str, ...] = (
    # Valuation ratios
    "be_me", "bm", "ev_", "sale_me", "div_", "eps_",

    # Profitability
    "gp_", "op_", "ni_", "ope_", "roe_", "roa_", "fcf_", "ebit",

    # Investment and growth
    "at_gr", "inv_gr", "capx_", "noa_gr",

    # Leverage and solvency
    "debt_", "lev_", "ltdebt_",
    
    # Accounting quality
    "accruals_", "tang_",
)

# K0 patterns identify fast-moving characteristics that are kept at the
# current value only. The pattern list is illustrative; final assignment
# defaults to K0 for any retained characteristic not matching the K1 patterns.
k0_factors: tuple[str, ...] = (
    "ret_", "rvol_", "ivol_", "beta_", "skew_",
    "kurt_", "mom_", "rmax_", "rmin_", "dolvol_",
    "turnover_", "amihud_", "iliq_", "zero_trades_",
)

metadata_cols = ("id", "date")

# Partition feature columns into K0 and K1 sets. Metadata columns pass through
# unchanged. Unmatched characteristics default to K0, which is the conservative
# assignment: it carries no incorrect temporal-structure assumption, though it
# may understate the lag value of some characteristics.
k1_cols = []
k0_cols = []
remaining = []

for col in feature_cols:
    if col in metadata_cols:
        continue
    if any(p in col for p in k1_factors):
        k1_cols.append(col)
    elif any(p in col for p in k0_factors):
        k0_cols.append(col)
    else:
        remaining.append(col)

k0_cols.extend(remaining)

print(f"K0(current only):{len(k0_cols):>4} characteristics")
print(f"K1(with annual lags):{len(k1_cols):>4} characteristics")
print(f"Unmatched(assigned K0):{len(remaining):>4}")

# Persist the K0 and K1 lists for the modelling notebook.
Path("..\\data\\k0_columns.json").write_text(pd.Series(k0_cols).to_json(orient="values"))
Path("..\\data\\k1_columns.json").write_text(pd.Series(k1_cols).to_json(orient="values"))

### Missing Data Handling

In [ ]:
# Load the cleaned feature panel and coerce the date column to datetime.
df = pd.read_parquet(FEATURES_RAW)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded panel shape: {df.shape}")

#TODO: Need to add some comments and details
SAMPLE_SIZE = 50_000
char_cols = []

for col in df.columns:
    if col in metadata_cols:
        continue
    series = df[col]
    if pd.api.types.is_datetime64_any_dtype(series):
        continue
    if not pd.api.types.is_numeric_dtype(series):
        continue

    values = series.dropna()
    if values.empty:
        continue

    sample = values.sample(SAMPLE_SIZE, random_state=0) if len(values) > SAMPLE_SIZE else values
    n_unique = sample.nunique()
    unique_ratio = n_unique / max(len(sample), 1)
    is_integer_like = bool(np.isclose(sample, sample.round()).all())
    is_binary = n_unique <= 2
    is_low_cardinality = n_unique < 50
    is_identifier_like = is_integer_like and unique_ratio > 0.80

    if is_binary or (is_integer_like and (is_low_cardinality or is_identifier_like)):
        continue

    char_cols.append(col)

print(f"Detected {len(char_cols)} characteristic columns out of {df.shape[1]} total columns.")